[Reference](https://ai.plainenglish.io/i-compared-4-python-ai-agent-frameworks-langgraph-won-clearly-f5adec0e1981$0)

# LangGraph: Built for Production


In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    query: str
    search_results: list
    summary: str
    iterations: Annotated[int, operator.add]
    is_complete: bool
def search_node(state: AgentState) -> AgentState:
    results = search_tool(state["query"])
    return {"search_results": results, "iterations": 1}
def summarize_node(state: AgentState) -> AgentState:
    summary = llm.invoke(f"Summarize: {state['search_results']}")
    return {"summary": summary.content}
def evaluate_node(state: AgentState) -> AgentState:
    evaluation = llm.invoke(
        f"Is this summary complete? Answer yes or no: {state['summary']}"
    )
    is_complete = "yes" in evaluation.content.lower()
    return {"is_complete": is_complete}
def should_continue(state: AgentState) -> str:
    if state["is_complete"] or state["iterations"] >= 3:
        return END
    return "search"
workflow = StateGraph(AgentState)
workflow.add_node("search", search_node)
workflow.add_node("summarize", summarize_node)
workflow.add_node("evaluate", evaluate_node)
workflow.set_entry_point("search")
workflow.add_edge("search", "summarize")
workflow.add_edge("summarize", "evaluate")
workflow.add_conditional_edges("evaluate", should_continue)
agent = workflow.compile()
result = agent.invoke({"query": "Python 3.14 free-threading", "iterations": 0})

# CrewAI: Fastest From Zero to Running


In [2]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()
researcher = Agent(
    role="Research Specialist",
    goal="Find accurate and recent information on the given topic",
    backstory="""You are an expert researcher with years of experience
    finding reliable information quickly and efficiently.""",
    tools=[search_tool],
    verbose=True
)
writer = Agent(
    role="Content Summarizer",
    goal="Create clear and comprehensive summaries from research findings",
    backstory="""You are a skilled writer who transforms complex research
    into clear, actionable summaries.""",
    verbose=True
)
research_task = Task(
    description="Search for recent information about {topic} and gather key findings",
    expected_output="A detailed list of key findings with sources",
    agent=researcher
)
summary_task = Task(
    description="Create a comprehensive summary from the research findings",
    expected_output="A clear, well-structured summary of 200-300 words",
    agent=writer,
    context=[research_task]
)
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, summary_task],
    process=Process.sequential,
    verbose=True
)
result = crew.kickoff(inputs={"topic": "Python 3.14 free-threading"})

# AutoGen / AG2: Powerful Conversation Engine

In [3]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

llm_config = {
    "config_list": [{"model": "gpt-4o", "api_key": "your-key"}],
    "temperature": 0.1
}
researcher = AssistantAgent(
    name="Researcher",
    system_message="""You are a research specialist. When given a topic,
    search for relevant information and present your findings clearly.
    Always cite your sources.""",
    llm_config=llm_config
)
summarizer = AssistantAgent(
    name="Summarizer",
    system_message="""You are a summarization expert. Take research findings
    and create concise, accurate summaries. If the summary is incomplete,
    ask the Researcher to search for more specific information.""",
    llm_config=llm_config
)
critic = AssistantAgent(
    name="Critic",
    system_message="""You evaluate summaries for completeness and accuracy.
    Reply TERMINATE only when the summary fully addresses the original query.""",
    llm_config=llm_config
)
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
)
groupchat = GroupChat(
    agents=[user_proxy, researcher, summarizer, critic],
    messages=[],
    max_round=10
)
manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)
user_proxy.initiate_chat(
    manager,
    message="Research and summarize: Python 3.14 free-threading performance"
)

# Pydantic AI: Lightweight and Type-Safe


In [4]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic import BaseModel
from typing import List

class ResearchResult(BaseModel):
    summary: str
    key_points: List[str]
    confidence_score: float
    needs_more_research: bool
class SearchTool(BaseModel):
    query: str
model = OpenAIModel("gpt-4o")
agent = Agent(
    model=model,
    result_type=ResearchResult,
    system_prompt="""You are a research assistant. Search for information
    on the given topic and provide a structured summary. Set needs_more_research
    to True if important information is missing."""
)
@agent.tool
async def search_web(ctx, search: SearchTool) -> str:
    results = await perform_search(search.query)
    return results
async def run_research_agent(topic: str) -> ResearchResult:
    max_iterations = 3
    current_query = topic
    for iteration in range(max_iterations):
        result = await agent.run(
            f"Research this topic: {current_query}",
        )
        if not result.data.needs_more_research:
            return result.data
        current_query = f"{topic} - focusing on missing aspects"
    return result.data
import asyncio
result = asyncio.run(run_research_agent("Python 3.14 free-threading"))
print(result.summary)
print(result.key_points)